# dee.cpp Ornith Milestone 3 forensic profile
This notebook is intentionally thin. It pins opt/real-model-t1 HEAD so
all M3 forensic instrumentation commits (02f80c6 persistent staging, 6ed324f
barrier removal, e5a610a device-resident MoE forward) plus the analyzer
instrumentation are present, runs the controlled matrix identically to
Milestone 2.5, and delegates analysis to analyze_milestone3_matrix.py.

In [ ]:
import importlib.metadata, json, os, platform, shutil, subprocess, sys, time
from pathlib import Path
import psutil, torch
print(json.dumps({
    'python': sys.version, 'platform': platform.platform(),
    'torch': torch.__version__, 'cuda_runtime': torch.version.cuda,
    'cpu_count': os.cpu_count(), 'ram_bytes': psutil.virtual_memory().total,
    'working_disk': shutil.disk_usage('/kaggle/working')._asdict(),
    'gpu_count': torch.cuda.device_count(),
    'gpus': [{'index': i, 'name': torch.cuda.get_device_name(i),
              'memory': torch.cuda.get_device_properties(i).total_memory}
             for i in range(torch.cuda.device_count())],
}, indent=2), flush=True)
subprocess.run(['nvidia-smi'], check=True)
assert torch.cuda.device_count() == 2, f'dual-T4 assignment required, got {torch.cuda.device_count()} GPUs'
assert all('T4' in torch.cuda.get_device_name(i) for i in range(2))

In [ ]:
import os, json
RUN_ID = os.environ.get('RUN_ID', 'LOCAL_RUN')
COMMIT_EXPECTED = os.environ.get('COMMIT_EXPECTED', '4d8ccf2')
print(json.dumps({'RUN_ID': RUN_ID, 'COMMIT_EXPECTED': COMMIT_EXPECTED}), flush=True)


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '-q',
                'transformers==5.14.1', 'safetensors==0.8.0',
                'pybind11==3.0.1', 'psutil==7.0.0', 'nvidia-ml-py'], check=True)
packages = {name: importlib.metadata.version(name) for name in
            ('transformers', 'safetensors', 'pybind11', 'psutil', 'nvidia-ml-py')}
print(json.dumps({'installed_packages': packages}, indent=2), flush=True)

In [ ]:
# Pin to opt/real-model-t1 HEAD so all post-M3 instrumentation commits
# (including path-proof counters, sync/overlap/multi-gpu helpers in
# run_ornith_forensics.py and the analyze_milestone3_matrix.py script)
# are present.  Checking out the floating branch HEAD (not a hard SHA)
# keeps the verifier authoritative on the latest M3 work.
ROOT = Path('/kaggle/temp/dee-source')
if ROOT.exists():
    assert str(ROOT.resolve()).startswith('/kaggle/temp/')
    shutil.rmtree(ROOT)
subprocess.run(['git', 'clone', '--branch', 'opt/real-model-t1', '--single-branch',
                'https://github.com/so-nerdyy/dee.git', str(ROOT)], check=True)
subprocess.run(['git', 'checkout', 'origin/opt/real-model-t1'], cwd=ROOT, check=True)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=ROOT, text=True).strip()
DEE = ROOT / 'dee.cpp'
EVIDENCE = Path(f'/kaggle/working/ornith-milestone3-evidence-{RUN_ID}')
EVIDENCE.mkdir(parents=True, exist_ok=True)
print({'repo': str(ROOT), 'commit': commit, 'evidence': str(EVIDENCE)}, flush=True)

In [ ]:
EXPECTED = COMMIT_EXPECTED
assert commit == EXPECTED, f'kernel preflight rejected: commit mismatch -- got {commit}, expected {EXPECTED}. The instrumentation target must be re-pinned.'
print(json.dumps({'preflight_1_commit_PASS': True, 'commit': commit}), flush=True)


In [ ]:
candidates = []
for index_path in Path('/kaggle/input').rglob('model.safetensors.index.json'):
    config_path = index_path.parent / 'config.json'
    if config_path.is_file() and json.loads(config_path.read_text()).get('model_type') == 'qwen3_5_moe':
        candidates.append(index_path.parent)
assert len(candidates) == 1, candidates
MODEL = candidates[0]
index = json.loads((MODEL / 'model.safetensors.index.json').read_text())
shards = sorted(set(index['weight_map'].values()))
assert len(shards) == 16 and all((MODEL / name).is_file() for name in shards)
print(json.dumps({'model_dir': str(MODEL), 'tensor_count': len(index['weight_map']),
                  'shard_count': len(shards),
                  'checkpoint_bytes': sum((MODEL / name).stat().st_size for name in shards)},
                 indent=2), flush=True)

In [ ]:
BUILD = DEE / 'build-kaggle-cuda'
subprocess.run(['cmake', '-S', str(DEE), '-B', str(BUILD), '-G', 'Ninja',
                '-DDEE_CUDA=ON', '-DDEE_BUILD_TESTS=ON',
                '-DCMAKE_CUDA_ARCHITECTURES=75', '-DCMAKE_BUILD_TYPE=Release'], check=True)
subprocess.run(['cmake', '--build', str(BUILD), '--parallel', '4'], check=True)
subprocess.run(['ctest', '--test-dir', str(BUILD), '--output-on-failure'], check=True)
subprocess.run([sys.executable, '-m', 'pytest',
                str(DEE / 'tests/test_milestone25_memory.py'),
                str(DEE / 'tests/test_analyze_milestone25_expert_trace.py'),
                str(DEE / 'tests/test_run_ornith_forensics.py'),
                str(DEE / 'tests/test_analyze_milestone25_matrix.py'),
                str(DEE / 'tests/test_analyze_milestone3_matrix.py'), '-q'],
               cwd=DEE, check=True)
env = os.environ.copy(); env['DEE_BUILD_DIR'] = str(BUILD)
subprocess.run([sys.executable, str(DEE / 'pydee/setup.py'), 'build_ext', '--inplace', '--force'],
               cwd=DEE, env=env, check=True)
print('CUDA build, native tests, Python tests, and pydee binding passed', flush=True)

In [ ]:
import os as _os
_so_candidates = list((DEE / 'pydee').glob('*.so'))
_so_candidates.extend(BUILD.glob('**/*.so'))
assert _so_candidates, 'Preflight #2 rejected: pydee or build-kaggle-cuda produced no .so'
_so = _so_candidates[0]
_git_index = ROOT / '.git' / 'index'
_mtime_diff = _os.path.getmtime(_so) - _os.path.getmtime(_git_index)
assert _mtime_diff > -1e-3, f'Preflight #2 rejected: built native extension is older than git checkout (mtime diff={_mtime_diff}s). The build did not run after the fresh checkout.'
print(json.dumps({'preflight_2_build_freshness_PASS': True, 'so_path': str(_so), 'mtime_diff_s': _mtime_diff}), flush=True)


In [ ]:
def run_tee(command, log_path):
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open('w', encoding='utf-8') as log:
        process = subprocess.Popen(command, cwd=DEE, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='', flush=True); log.write(line); log.flush()
        code = process.wait()
    if code:
        raise subprocess.CalledProcessError(code, command)

# Reusing the existing run_milestone25_matrix.py driver.  Its 9-experiment
# matrix produces identical raw forensic JSONs to M2.5; the deltas vs
# M2.5 are produced later by analyze_milestone3_matrix.py.
matrix_command = [sys.executable, '-u', '-X', 'faulthandler',
                  str(DEE / 'scripts/run_milestone25_matrix.py'),
                  '--model-dir', str(MODEL), '--output-dir', str(EVIDENCE),
                  '--require-dual-gpu', '--skip-aggregate',
                  '--kernel-slug', 'nivind/dee-cpp-ornith-milestone-3-forensics']
run_tee(matrix_command, EVIDENCE / 'logs/matrix-driver.log')
print('Controlled matrix finished; cold run was first', flush=True)

In [ ]:
router_report = EVIDENCE / 'ornith-router-parity.json'
subprocess.run([sys.executable, '-u', '-X', 'faulthandler',
                str(DEE / 'scripts/run_ornith_router_parity.py'),
                '--model-dir', str(MODEL), '--layers', '0', '3', '20', '39',
                '--tokens', '16', '--report', str(router_report)], cwd=DEE, check=True)
router = json.loads(router_report.read_text()); assert router['result'] == 'PASS'
assert all(item['expert_ids_exact'] for item in router['layers'])
layer0_report = EVIDENCE / 'ornith-layer0-regression.json'
subprocess.run([sys.executable, '-u', '-X', 'faulthandler',
                str(DEE / 'scripts/run_ornith_layer0_parity.py'),
                '--model-dir', str(MODEL), '--max-prompt-tokens', '4',
                '--report', str(layer0_report)], cwd=DEE, check=True)
layer0 = json.loads(layer0_report.read_text()); assert layer0['pass'] is True
print({'router': router['result'], 'layer0': layer0['pass']}, flush=True)

In [ ]:
# The M2.5 analyzer would compare rounds to M2.5 baseline; for M3 we
# invoke the dedicated M3 comparator that reads path-proof.json, sync,
# overlap, multi-gpu timeline and emits MILESTONE_3_VERIFICATION.md
# plus the documented M3 deliverable list.
analysis_command = [sys.executable, '-u', str(DEE / 'scripts/analyze_milestone3_matrix.py'),
                    '--m3-dir', str(EVIDENCE),
                    '--m25-dir', str(DEE / 'benchmark_reports/milestone-2.5/kaggle-forensics-latest-output/ornith-milestone25-evidence'),
                    '--output-dir', str(EVIDENCE / 'analysis')]
run_tee(analysis_command, EVIDENCE / 'logs/final-analysis.log')
final_report = json.loads((EVIDENCE / 'analysis/milestone-3-report.json').read_text())
assert final_report['result'] == 'PASS', final_report['result']
summary = final_report['defects_summary']
failures = [label for label, count in summary.items() if label not in ('fully_fixed', 'partially_fixed', 'unavoidable') and count > 0]
assert not failures, {'forensic_result': final_report['result'], 'non_compliant': failures}
print({'forensic_result': final_report['result'], 'defects_summary': summary}, flush=True)

In [ ]:
archive = shutil.make_archive('/kaggle/working/ornith-milestone3-evidence', 'gztar',
                              root_dir=EVIDENCE.parent, base_dir=EVIDENCE.name)
required = ['MILESTONE_3_VERIFICATION.md', 'milestone-3-report.json',
            'before-after-milestone25.json', 'acceptance-audit.json',
            'correctness-report.json', 'environment.json', 'matrix-summary.json',
            'memory-timeline.json', 'layer-timing.json',
            'transfer-analysis.json', 'synchronization-analysis.json',
            'overlap-analysis.json', 'multi-gpu-timeline.json',
            'path-proof.json', 'profiler-summary.md', 'bottleneck-ranking.json',
            'evidence-integrity-sha256.txt']
present = {name: (EVIDENCE / 'analysis' / name).is_file()
           if name in ('MILESTONE_3_VERIFICATION.md', 'milestone-3-report.json',
                       'before-after-milestone25.json', 'acceptance-audit.json',
                       'correctness-report.json', 'environment.json',
                       'matrix-summary.json', 'memory-timeline.json',
                       'layer-timing.json', 'transfer-analysis.json',
                       'synchronization-analysis.json', 'overlap-analysis.json',
                       'multi-gpu-timeline.json', 'path-proof.json',
                       'profiler-summary.md', 'bottleneck-ranking.json',
                       'evidence-integrity-sha256.txt') else (EVIDENCE / name).is_file()
           for name in required}
missing = [name for name, ok in present.items() if not ok]
assert not missing, f'missing required files: {missing}'
print(json.dumps({'final_status': 'PASS', 'archive': archive,
                  'required_files_meta': {name: (EVIDENCE / 'analysis' / name).stat().st_size
                                            if (EVIDENCE / 'analysis' / name).is_file() else 0
                                            for name in required}},
                 indent=2), flush=True)